In [18]:
# -------------------------------------------------
# Cell 1 – Import, configurazione e inizializzazione client
# -------------------------------------------------
import os
import pyaudio
import threading
import io
import wave
from huggingface_hub import InferenceClient
import requests

# ---------- Audio parameters ----------
FORMAT = pyaudio.paInt16          # 16‑bit PCM
CHANNELS = 1                      # mono mic
RATE = 16000                      # 16 kHz (common for Whisper)
CHUNK = 1024                      # buffer size

# ---------- Model and provider configuration ----------
MODEL_NAME = "openai/whisper-large-v3"

# Lista di provider da provare (in ordine di preferenza)
PROVIDERS_TO_TRY = ["fal-ai", "together", "openai", "hf-inference", "auto"]

# ---------- Initialize client with best provider ----------
client = None
selected_provider = None

try:
    client = InferenceClient(
        model=MODEL_NAME,
        provider="fal-ai",
        token=os.getenv("HF_TOKEN_READ", None)
    )
    selected_provider = "fal-ai"
    print("✅ InferenceClient inizializzato con provider 'fal-ai'.")
    
except Exception as e:
    print(f"❌ Impossibile inizializzare InferenceClient con 'together': {e}")
    print("🔄 Proveremo con altri provider durante la trascrizione.")
    
    # Se together non funziona, inizializza un client generico che userà fallback
    try:
        client = InferenceClient(
            model=MODEL_NAME,
            provider="fal-ai",
            token=os.getenv("HF_TOKEN_READ", None)
        )
        selected_provider = "fal-ai"
        print("✅ InferenceClient inizializzato con provider 'fal-ai'.")
    except Exception as auto_error:
        print(f"❌ Impossibile inizializzare InferenceClient: {auto_error}")
        client = None
        selected_provider = None

print(f"📋 Modello: {MODEL_NAME}")
print(f"🔧 Provider selezionato: {selected_provider if selected_provider else 'Nessuno'}")

✅ InferenceClient inizializzato con provider 'fal-ai'.
📋 Modello: openai/whisper-large-v3
🔧 Provider selezionato: fal-ai


In [19]:
# -------------------------------------------------
# Cell 2 – Stato globale, thread di registrazione e funzioni di controllo
# -------------------------------------------------
audio_state = {
    "is_recording": False,
    "frames": [],
    "stream": None,
    "p": pyaudio.PyAudio(),
    "thread": None
}

def _recording_thread_logic():
    """Thread che legge dal microfono e salva i campioni."""
    try:
        audio_state["stream"] = audio_state["p"].open(
            format=FORMAT,
            channels=CHANNELS,
            rate=RATE,
            input=True,
            frames_per_buffer=CHUNK,
        )
        print("🎤 Microfono aperto e pronto a registrare.")
    except Exception as e:
        print(f"❌ Impossibile aprire lo stream audio: {e}")
        print("Verifica che il microfono sia collegato e funzionante.")
        audio_state["is_recording"] = False
        return

    audio_state["frames"] = []
    frame_count = 0
    
    while audio_state["is_recording"]:
        try:
            data = audio_state["stream"].read(CHUNK, exception_on_overflow=False)
            audio_state["frames"].append(data)
            frame_count += 1
            
            # Mostra progresso ogni 100 frame (circa 6.4 secondi a 16kHz)
            if frame_count % 100 == 0:
                duration = (frame_count * CHUNK) / RATE
                print(f"⏱️ Registrazione in corso... {duration:.1f}s")
                
        except Exception as e:
            print(f"⚠️ Errore durante la lettura: {e}")
            break

    # Chiusura pulita dello stream
    if audio_state["stream"]:
        audio_state["stream"].stop_stream()
        audio_state["stream"].close()
        audio_state["stream"] = None
    
    total_duration = (len(audio_state["frames"]) * CHUNK) / RATE
    print(f"✅ Registrazione terminata. Durata totale: {total_duration:.1f}s")

def start_recording():
    """Avvia la registrazione."""
    if audio_state["is_recording"]:
        print("⏩ La registrazione è già in corso.")
        return
    
    audio_state["is_recording"] = True
    audio_state["thread"] = threading.Thread(target=_recording_thread_logic, daemon=True)
    audio_state["thread"].start()
    print("🔴 Registrazione avviata – scrivi 'stop' per fermarla.")

def stop_recording():
    """Ferma la registrazione."""
    if not audio_state["is_recording"]:
        print("⏸️ Nessuna registrazione attiva.")
        return
    
    audio_state["is_recording"] = False
    if audio_state["thread"]:
        audio_state["thread"].join()
    audio_state["thread"] = None

def save_audio_to_file(frames, filename=None):
    """Salva l'audio registrato in un file WAV (opzionale)."""
    if not frames:
        print("⚠️ Nessun audio da salvare.")
        return None
    
    if filename is None:
        from datetime import datetime
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"audio_registrato_{timestamp}.wav"
    
    try:
        with wave.open(filename, 'wb') as wf:
            wf.setnchannels(CHANNELS)
            wf.setsampwidth(audio_state["p"].get_sample_size(FORMAT))
            wf.setframerate(RATE)
            wf.writeframes(b''.join(frames))
        
        import os
        file_size = os.path.getsize(filename)
        duration = (len(frames) * CHUNK) / RATE
        
        print(f"💾 Audio salvato: {filename} ({file_size:,} bytes, {duration:.1f}s)")
        return filename
        
    except Exception as e:
        print(f"❌ Errore nel salvare il file: {e}")
        return None

print("✅ Funzioni di registrazione definite.")


✅ Funzioni di registrazione definite.


In [20]:
# -------------------------------------------------
# Cell 3 – Funzione di trascrizione con sistema di fallback
# -------------------------------------------------
def transcribe_with_fallback():
    """
    Trascrive l'audio provando automaticamente provider diversi.
    """
    if not audio_state["frames"]:
        print("⚠️ Nessun audio da trascrivere.")
        return

    # Creazione del buffer WAV in memoria
    print("⏳ Preparazione del buffer audio...")
    wav_buf = io.BytesIO()
    with wave.open(wav_buf, "wb") as wf:
        wf.setnchannels(CHANNELS)
        wf.setsampwidth(audio_state["p"].get_sample_size(FORMAT))
        wf.setframerate(RATE)
        wf.writeframes(b''.join(audio_state["frames"]))
    wav_buf.seek(0)

    # Informazioni di debug sul buffer
    audio_data = wav_buf.getvalue()
    print(f"🔍 Dimensione buffer: {len(audio_data):,} bytes")
    print(f"🔍 Header WAV: {audio_data[:12]}")
    
    # Lista di provider da provare
    providers_to_try = PROVIDERS_TO_TRY.copy()
    
    # Se abbiamo già un provider che funziona, provalo per primo
    if selected_provider and selected_provider in providers_to_try:
        providers_to_try.remove(selected_provider)
        providers_to_try.insert(0, selected_provider)
    
    print(f"🚀 Iniziando tentativi di trascrizione con provider: {providers_to_try}")
    
    success = False
    
    for provider in providers_to_try:
        print(f"\n🔄 Tentativo con provider: {provider}")
        try:
            if client is None:
                raise RuntimeError("InferenceClient non inizializzato.")
            
            # Crea un client con il provider specifico
            test_client = InferenceClient(
                model=MODEL_NAME,
                provider=provider,
                token=os.getenv("HF_TOKEN_READ", None)
            )
            
            # Tenta la trascrizione
            print(f"⏳ Chiamata API in corso...")
            result = test_client.automatic_speech_recognition(
                audio=wav_buf.getvalue()
            )
            
            testo = result.get("text", "")
            if testo:
                print(f"\n--- TRASCRIZIONE (provider: {provider}) ---")
                print(testo)
                print("--- FINE ---")
                success = True
                break
            else:
                print(f"⚠️ Provider {provider} ha restituito testo vuoto")
                
        except Exception as e:
            error_msg = str(e)
            print(f"❌ Provider {provider} fallito: {error_msg[:200]}")
            
            # Se l'errore è di provider non supportato, continua
            if "not supported" in error_msg.lower():
                continue
            # Se è un errore di modello non trovato, continua
            elif "not found" in error_msg.lower():
                continue
            # Se è un errore di autenticazione, prova il prossimo
            elif "auth" in error_msg.lower() or "token" in error_msg.lower():
                continue
            else:
                # Altri errori potrebbero essere temporanei, continua comunque
                continue
    
    if not success:
        print("\n❌ NESSUN PROVIDER È RIUSCITO A TRASCRIVERE L'AUDIO")
        print("💡 Possibili cause:")
        print("   - Il modello non è disponibile via API")
        print("   - Problemi di autenticazione")
        print("   - Problemi di quota/limiti")
        print("   - Formato audio non supportato")
        
        # Offri di salvare l'audio per debug
        save_choice = input("🔍 Vuoi salvare l'audio per debug? (y/n): ").strip().lower()
        if save_choice in ['y', 'yes', 's', 'si']:
            save_audio_to_file(audio_state["frames"])
    
    # Pulizia del buffer
    audio_state["frames"] = []
    return success

print("✅ Funzione di trascrizione con fallback definita.")


✅ Funzione di trascrizione con fallback definita.


In [22]:
# -------------------------------------------------
# Cell 4 – Loop di controllo interattivo
# -------------------------------------------------
def main():
    """Loop principale per il controllo della registrazione e trascrizione."""
    print("\n🎤 SISTEMA DI TRASCRIZIONE AUDIO")
    print("=" * 50)
    print("🤖 Modello:", MODEL_NAME)
    print("🔧 Provider configurati:", PROVIDERS_TO_TRY)
    print("=" * 50)
    print("Comandi disponibili:")
    print("  start   → avvia la registrazione")
    print("  stop    → ferma la registrazione e trascrive")
    print("  save    → salva l'ultima registrazione su file")
    print("  status  → mostra lo stato attuale")
    print("  debug   → esegue diagnostiche di sistema")
    print("  exit    → esce dal programma")
    print("=" * 50)
    
    while True:
        try:
            cmd = input("\n🔹 Comando (start/stop/save/status/debug/exit): ").strip().lower()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Uscita dal programma.")
            break
        
        if cmd == "start":
            start_recording()
            
        elif cmd == "stop":
            print("⏹️ Fermando la registrazione...")
            stop_recording()
            print("🎯 Iniziando la trascrizione...")
            transcribe_with_fallback()
            
        elif cmd == "save":
            if audio_state["frames"]:
                save_audio_to_file(audio_state["frames"])
            else:
                print("⚠️ Nessuna registrazione da salvare.")
                
        elif cmd == "status":
            print(f"\n📊 STATO ATTUALE:")
            print(f"   🎤 Registrazione attiva: {'Sì' if audio_state['is_recording'] else 'No'}")
            print(f"   📝 Frame registrati: {len(audio_state['frames'])}")
            print(f"   🤖 Client inizializzato: {'Sì' if client else 'No'}")
            print(f"   🔧 Provider corrente: {selected_provider}")
            print(f"   ⏱️ Durata stimata registrazione: {len(audio_state['frames']) * CHUNK / RATE:.1f}s")
            
        elif cmd == "debug":
            print("\n🔍 ESEGUENDO DIAGNOSTICHE...")
            
            # Test del client
            if client:
                print("   ✅ Client InferenceClient OK")
            else:
                print("   ❌ Client InferenceClient non inizializzato")
            
            # Test del token
            token = os.getenv("HF_TOKEN_READ")
            if token:
                print(f"   ✅ Token HF_TOKEN_READ trovato ({len(token)} caratteri)")
            else:
                print("   ⚠️ Token HF_TOKEN_READ non trovato (userà login CLI)")
            
            # Test dispositivo audio
            try:
                device_info = audio_state["p"].get_default_input_device_info()
                print(f"   🎤 Dispositivo audio: {device_info['name']}")
            except:
                print("   ⚠️ Impossibile accedere al dispositivo audio")
            
            print("   ✅ Diagnostiche complete")
            
        elif cmd == "exit":
            if audio_state["is_recording"]:
                print("⚠️ Registrazione ancora attiva. Ferma prima di uscire.")
                continue
            print("👋 Uscita dal programma.")
            break
            
        else:
            print("❓ Comando non riconosciuto. Usa: start, stop, save, status, debug o exit")

print("✅ Loop di controllo definito.")


✅ Loop di controllo definito.


In [23]:
# -------------------------------------------------
# Cell 5 – Avvio del sistema di trascrizione
# -------------------------------------------------
print("🚀 SISTEMA PRONTO!")
print(f"📋 Modello: {MODEL_NAME}")
print(f"🔧 Provider: {selected_provider if selected_provider else 'Non configurato'}")
print("🔧 Provider di fallback:", PROVIDERS_TO_TRY)

if client:
    print("\n✅ Tutto configurato correttamente.")
    print("💡 Suggerimento: esegui 'debug' per verificare lo stato del sistema.")
else:
    print("\n⚠️ Client non inizializzato. Alcune funzioni potrebbero non funzionare.")
    print("💡 Suggerimento: verifica la configurazione del token HF_TOKEN_READ.")

print("\n🎯 Digita 'start' per iniziare la registrazione.")

# Avvia il loop principale
main()


🚀 SISTEMA PRONTO!
📋 Modello: openai/whisper-large-v3
🔧 Provider: fal-ai
🔧 Provider di fallback: ['fal-ai', 'together', 'openai', 'hf-inference', 'auto']

✅ Tutto configurato correttamente.
💡 Suggerimento: esegui 'debug' per verificare lo stato del sistema.

🎯 Digita 'start' per iniziare la registrazione.

🎤 SISTEMA DI TRASCRIZIONE AUDIO
🤖 Modello: openai/whisper-large-v3
🔧 Provider configurati: ['fal-ai', 'together', 'openai', 'hf-inference', 'auto']
Comandi disponibili:
  start   → avvia la registrazione
  stop    → ferma la registrazione e trascrive
  save    → salva l'ultima registrazione su file
  status  → mostra lo stato attuale
  debug   → esegue diagnostiche di sistema
  exit    → esce dal programma



🔹 Comando (start/stop/save/status/debug/exit):  debug



🔍 ESEGUENDO DIAGNOSTICHE...
   ✅ Client InferenceClient OK
   ✅ Token HF_TOKEN_READ trovato (37 caratteri)
   🎤 Dispositivo audio: Microphone (High Definition Aud
   ✅ Diagnostiche complete



🔹 Comando (start/stop/save/status/debug/exit):  start


🔴 Registrazione avviata – scrivi 'stop' per fermarla.
🎤 Microfono aperto e pronto a registrare.
⏱️ Registrazione in corso... 6.4s
⏱️ Registrazione in corso... 12.8s
⏱️ Registrazione in corso... 19.2s
⏱️ Registrazione in corso... 25.6s
⏱️ Registrazione in corso... 32.0s
⏱️ Registrazione in corso... 38.4s
⏱️ Registrazione in corso... 44.8s
⏱️ Registrazione in corso... 51.2s
⏱️ Registrazione in corso... 57.6s
⏱️ Registrazione in corso... 64.0s
⏱️ Registrazione in corso... 70.4s


...

⏹️ Fermando la registrazione...
✅ Registrazione terminata. Durata totale: 74.4s
🎯 Iniziando la trascrizione...
⏳ Preparazione del buffer audio...
🔍 Dimensione buffer: 2,379,820 bytes
🔍 Header WAV: b'RIFF$P$\x00WAVE'
🚀 Iniziando tentativi di trascrizione con provider: ['fal-ai', 'together', 'openai', 'hf-inference', 'auto']

🔄 Tentativo con provider: fal-ai
⏳ Chiamata API in corso...

--- TRASCRIZIONE (provider: fal-ai) ---
 Sto cercando di riavviare la registrazione nella speranza di riuscire a registrare più di 30 secondi di vocale e a questo punto risulta importante verificare attraverso il timing della registrazione in corso che effettivamente siano stati siano trascorsi più di 30 secondi. Non amo particolarmente il prosciutto, il mio salume preferito sicuramente non è il prosciutto, però negli anni sono passato da mangiare prevalentemente il prosciutto cotto da ragazzino al mangiare anche e soprattutto il prosciutto crudo, il prosciutto crudo di montagna, il prosciutto crudo stagio


🔹 Comando (start/stop/save/status/debug/exit):  stop


⏹️ Fermando la registrazione...
⏸️ Nessuna registrazione attiva.
🎯 Iniziando la trascrizione...
⚠️ Nessun audio da trascrivere.



🔹 Comando (start/stop/save/status/debug/exit):  exit


👋 Uscita dal programma.
